# 02. Відбір за CPV-кодами
**Мета:** виділити закупівлі БпЛА і РЕБ з 12.9 млн тендерів.

**Вхід:** `data/raw/tenders_*.csv`.

**Що робимо:** категорія за головним CPV: 3471 БпЛА, 3573 РЕБ (ядро), 3572 Розвідка і виявлення, 3571 Керування і зв'язок (контекст); кількість за категорією і процедурою.

**Вихід:** `data/processed/candidates.parquet` (15 532 тендери).

In [1]:
import duckdb
con = duckdb.connect()
path = '../data/raw/'

In [2]:
con.sql(f"""
    SELECT
        CASE
            WHEN main_cpv_code LIKE '3471%' THEN 'БпЛА'
            WHEN main_cpv_code LIKE '3573%' THEN 'РЕБ'
            WHEN main_cpv_code LIKE '3572%' THEN 'Розвідка і виявлення'
            WHEN main_cpv_code LIKE '3571%' THEN 'Керування і зв''язок'
        END AS category,
        procurement_method,
        COUNT(*) AS n
    FROM '{path}tenders_*.csv'
    WHERE main_cpv_code LIKE '3471%'
       OR main_cpv_code LIKE '3571%'
       OR main_cpv_code LIKE '3572%'
       OR main_cpv_code LIKE '3573%'
    GROUP BY ALL
    ORDER BY category, n DESC
""").show(max_rows=50)

┌──────────────────────┬────────────────────┬───────┐
│       category       │ procurement_method │   n   │
│       varchar        │      varchar       │ int64 │
├──────────────────────┼────────────────────┼───────┤
│ БпЛА                 │ selective          │  5195 │
│ БпЛА                 │ limited            │  4489 │
│ БпЛА                 │ open               │  2925 │
│ Керування і зв'язок  │ limited            │   155 │
│ Керування і зв'язок  │ open               │    49 │
│ РЕБ                  │ limited            │  1307 │
│ РЕБ                  │ open               │  1054 │
│ Розвідка і виявлення │ limited            │   276 │
│ Розвідка і виявлення │ open               │    82 │
└──────────────────────┴────────────────────┴───────┘



In [3]:
con.sql(f"""
    COPY (
        SELECT
            CASE
                WHEN main_cpv_code LIKE '3471%' THEN 'БпЛА'
                WHEN main_cpv_code LIKE '3573%' THEN 'РЕБ'
                WHEN main_cpv_code LIKE '3572%' THEN 'Розвідка і виявлення'
                WHEN main_cpv_code LIKE '3571%' THEN 'Керування і зв''язок'
            END AS category,
            *
        FROM '{path}tenders_*.csv'
        WHERE main_cpv_code LIKE '3471%'
           OR main_cpv_code LIKE '3571%'
           OR main_cpv_code LIKE '3572%'
           OR main_cpv_code LIKE '3573%'
    ) TO '../data/processed/candidates.parquet' (FORMAT parquet)
""")

In [4]:
con.sql("""
    SELECT category, COUNT(*) AS n
    FROM '../data/processed/candidates.parquet'
    GROUP BY category
    ORDER BY n DESC
""").show()

┌──────────────────────┬───────┐
│       category       │   n   │
│       varchar        │ int64 │
├──────────────────────┼───────┤
│ БпЛА                 │ 12609 │
│ РЕБ                  │  2361 │
│ Розвідка і виявлення │   358 │
│ Керування і зв'язок  │   204 │
└──────────────────────┴───────┘

